# 01 - Data collection (Stage 1)
**Owner:** Aiman | **Calendar:** Week 8 (5-11 Oct) | **Branch:** `aiman/data`

This notebook only **downloads and inspects** the raw data. It does not preprocess (that is notebook 02).

What it does:
1. Checks Kaggle authentication
2. Downloads APTOS 2019 to local Colab disk (`/content/data/raw`, **not** Drive)
3. Inspects folder layout, finds label CSVs, matches labels to images, prints class counts
4. Checks the EyePACS mirror size, then downloads it only if you switch the flag on
5. Saves a small `raw_inventory.json` to the shared Drive for Maheen and Amna

Before running: edit the 4 values at the top of the next cell (copy them from notebook 00), set `BRANCH = "aiman/data"`, and make sure the Colab Secrets exist (`GITHUB_TOKEN` and `KAGGLE_API_TOKEN`, or the legacy `KAGGLE_USERNAME` + `KAGGLE_KEY`).

In [ ]:
# ===== X-Retina bootstrap: paste as the FIRST cell of every notebook =====
# Edit the 4 values below once. Needs Colab Secrets: GITHUB_TOKEN, KAGGLE_USERNAME, KAGGLE_KEY
GITHUB_USER = "YOUR_GITHUB_USERNAME"   # owner of the repo
REPO_NAME   = "x-retina"
BRANCH      = "main"                   # or your own branch, e.g. "aiman/data"
GIT_NAME    = "Your Name"              # used for commits made from Colab
GIT_EMAIL   = "you@example.com"

import os, sys, json, subprocess
from google.colab import drive, userdata

# 1) Drive + shared folder
drive.mount("/content/drive")
DRIVE_ROOT = "/content/drive/MyDrive/XRetina_Shared"
assert os.path.isdir(DRIVE_ROOT), (
    f"Shared folder not found at {DRIVE_ROOT}. Add a shortcut named 'XRetina_Shared' to My Drive.")
for _sub in ("processed", "splits", "checkpoints", "outputs"):
    os.makedirs(f"{DRIVE_ROOT}/{_sub}", exist_ok=True)
os.environ["XRETINA_DRIVE_ROOT"] = DRIVE_ROOT
os.makedirs("/content/data/raw", exist_ok=True)
os.environ["XRETINA_RAW_DIR"] = "/content/data/raw"

# 2) Clone or update the repo (token is hidden from error messages)
_token = userdata.get("GITHUB_TOKEN")
REPO_DIR = f"/content/{REPO_NAME}"
_url = f"https://{_token}@github.com/{GITHUB_USER}/{REPO_NAME}.git"

def _run(cmd):
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(f"{' '.join(cmd[:4])} failed:\n{r.stderr.replace(_token, '***')}")
    return r.stdout.strip()

if not os.path.isdir(REPO_DIR):
    _run(["git", "clone", _url, REPO_DIR])
_run(["git", "-C", REPO_DIR, "remote", "set-url", "origin", _url])
_run(["git", "-C", REPO_DIR, "fetch", "origin"])
try:
    _run(["git", "-C", REPO_DIR, "checkout", BRANCH])
except RuntimeError:
    _run(["git", "-C", REPO_DIR, "checkout", "-b", BRANCH])  # first time on a new branch
try:
    _run(["git", "-C", REPO_DIR, "pull", "origin", BRANCH])
except RuntimeError as e:
    print("Pull skipped (new local branch or local changes):", str(e).splitlines()[-1])
_run(["git", "-C", REPO_DIR, "config", "user.name", GIT_NAME])
_run(["git", "-C", REPO_DIR, "config", "user.email", GIT_EMAIL])

# 3) Requirements
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{REPO_DIR}/requirements.txt"], check=True)

# 4) Kaggle credentials: new token (KAGGLE_API_TOKEN, preferred) or legacy KAGGLE_USERNAME + KAGGLE_KEY
_kaggle_mode = None
try:
    os.environ["KAGGLE_API_TOKEN"] = userdata.get("KAGGLE_API_TOKEN")
    _kaggle_mode = "token"
except Exception:
    try:
        os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
        _kp = os.path.expanduser("~/.kaggle/kaggle.json")
        with open(_kp, "w") as f:
            json.dump({"username": userdata.get("KAGGLE_USERNAME"), "key": userdata.get("KAGGLE_KEY")}, f)
        os.chmod(_kp, 0o600)
        _kaggle_mode = "legacy kaggle.json"
    except Exception:
        print("No Kaggle secrets found (only needed for dataset download).")
print("Kaggle auth mode:", _kaggle_mode)

# 5) Make `import src...` work
os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

def git_push(message):
    """Commit and push your work from Colab. Refuses to push to main (use a PR)."""
    if BRANCH == "main":
        raise RuntimeError("Do not push to main. Set BRANCH to your own branch (e.g. 'aiman/data') and re-run the bootstrap.")
    _run(["git", "-C", REPO_DIR, "add", "-A"])
    print(_run(["git", "-C", REPO_DIR, "commit", "-m", message]))
    print(_run(["git", "-C", REPO_DIR, "push", "-u", "origin", BRANCH]))

from src.config import get_paths
from src.utils import seed_everything, get_device
PATHS = get_paths()
print("Repo:", REPO_DIR, "| Branch:", BRANCH, "| Device:", get_device())
print({k: str(v) for k, v in PATHS.items()})


## 1. Kaggle authentication check

In [ ]:
from src.data.download import check_auth, dataset_size, download_dataset, describe_tree, free_disk_gb
from src.data.labels import load_labels, class_table, patient_id_from_name
from src.config import load_yaml
import json, pandas as pd
import matplotlib.pyplot as plt

DS = load_yaml("datasets.yaml")
CLASS_NAMES = DS["classes"]
check_auth()
print(f"Free local disk: {free_disk_gb():.1f} GB")

## 2. APTOS 2019
The page for this mirror lists about 8.6 GB. The download takes several minutes; it is skipped automatically if it already finished in this session.

In [ ]:
aptos_slug = DS["aptos2019"]["kaggle_slug"]
line, size_gb = dataset_size(aptos_slug)
print("Listing:", line)
print("Size (GB):", size_gb)

APTOS_DIR = PATHS["raw"] / DS["aptos2019"]["raw_subdir"]
download_dataset(aptos_slug, APTOS_DIR)

### 2b. Inspect APTOS
This finds the label CSVs and matches each label to an image file. It does not assume a folder layout.

In [ ]:
describe_tree(APTOS_DIR, max_depth=2)
aptos_df, aptos_rep = load_labels(APTOS_DIR)
display(aptos_df.head())
display(aptos_df.groupby("source_csv")["label"].value_counts().unstack(fill_value=0))
aptos_tbl = class_table(aptos_df, CLASS_NAMES)
display(aptos_tbl)

In [ ]:
# Sanity checks (these should all pass before moving on)
print("Images found for labels:", len(aptos_df))
assert aptos_rep["labeled_without_image"] == 0 or aptos_rep["labeled_without_image"] < 0.01 * len(aptos_df), "Many labels have no image - check the layout above"
assert aptos_df["label"].between(0, 4).all()
print("Expected: about 3,662 labeled images if the mirror holds the full competition train set.")

ax = aptos_tbl.set_index("name")["count"].plot(kind="bar", rot=20, title="APTOS 2019 - images per DR grade")
ax.set_ylabel("images"); plt.tight_layout(); plt.show()

## 3. EyePACS
**Check size first.** The full Kaggle EyePACS data is tens of GB. Colab's free disk is limited, and unzipping needs room for both the archive and the extracted images.

1. Run the next cell and read the printed size.
2. If it fits (the cell tells you), set `DOWNLOAD_EYEPACS = True` and re-run.
3. If it is too big, do not force it. Tell me and we will use a smaller source or run this stage on Kaggle's own notebooks.

In [ ]:
DOWNLOAD_EYEPACS = False   # <- set True only after reading the size below

eye_slug = DS["eyepacs"]["kaggle_slug"]
line, eye_gb = dataset_size(eye_slug)
print("Listing:", line)
print("Size (GB):", eye_gb, "| free disk (GB):", round(free_disk_gb(), 1))
if eye_gb is not None:
    print("Fits (needs ~2.2x for download + unzip):", eye_gb * 2.2 < free_disk_gb())
else:
    print("Size unknown - the slug may be wrong or the listing is hidden. Check configs/datasets.yaml.")

EYE_DIR = PATHS["raw"] / DS["eyepacs"]["raw_subdir"]
if DOWNLOAD_EYEPACS:
    download_dataset(eye_slug, EYE_DIR)

In [ ]:
if DOWNLOAD_EYEPACS:
    describe_tree(EYE_DIR, max_depth=2)
    eye_df, eye_rep = load_labels(EYE_DIR)
    eye_df["patient_id"] = eye_df["image_id"].map(patient_id_from_name)
    print("\nImages per patient (EyePACS has left + right eye per patient):")
    print("  images:", len(eye_df), "| patients:", eye_df["patient_id"].nunique(), "| missing patient id:", int(eye_df["patient_id"].isna().sum()))
    eye_tbl = class_table(eye_df, CLASS_NAMES)
    display(eye_tbl)
    ax = eye_tbl.set_index("name")["count"].plot(kind="bar", rot=20, title="EyePACS - images per DR grade")
    ax.set_ylabel("images"); plt.tight_layout(); plt.show()
else:
    print("EyePACS not downloaded yet (DOWNLOAD_EYEPACS = False).")

## 4. Save a raw inventory for the team
Small JSON on the shared Drive so Maheen and Amna can see what was downloaded without re-running anything.

In [ ]:
inventory = {"aptos2019": {"slug": aptos_slug, "report": {k: (v if not isinstance(v, list) else [str(x) for x in v]) for k, v in aptos_rep.items()},
                           "class_counts": aptos_tbl.set_index("name")["count"].to_dict()}}
if DOWNLOAD_EYEPACS:
    inventory["eyepacs"] = {"slug": eye_slug, "report": {k: (v if not isinstance(v, list) else [str(x) for x in v]) for k, v in eye_rep.items()},
                            "class_counts": eye_tbl.set_index("name")["count"].to_dict(),
                            "n_patients": int(eye_df["patient_id"].nunique())}
out = PATHS["outputs"] / "raw_inventory.json"
out.write_text(json.dumps(inventory, indent=2, default=str))
print("Saved:", out)

## Done check - Week 8, part 1
- [ ] Kaggle auth OK
- [ ] APTOS downloaded; labeled images matched; class counts printed
- [ ] EyePACS size checked (and downloaded if it fits)
- [ ] `raw_inventory.json` saved to the shared Drive

**Next:** notebook 02 (circular crop, CLAHE, resize, stratified splits, `manifest.csv`). Wait for the go-ahead.

To save this notebook: `git_push("Add notebook 01 data collection")` (works because `BRANCH = "aiman/data"`), then open a Pull Request into `main`.